# Athletes MLOps Pipeline, Notebook Runner

ADSP 31021 Assignment 2. Runs the full pipeline: clean, featurize, Feast
feature store, four MLflow experiments on Databricks, and evaluation.

This notebook works in **Google Colab** or a **local Jupyter** session. It runs
the same `src/` scripts as the shell path, so results are identical.

You will need your Databricks workspace URL and a personal access token.


## 1. Get the code

In Colab, clone the repo. Locally (if you already have the repo open), skip
this cell and just make sure the notebook runs from the repo root.


In [ ]:
# Colab: clone your repository. Replace the URL with your repo.
# Skip this cell if running locally from inside the repo.
import os

REPO_URL = 'https://github.com/mwaseemUC/athletes-mlops-feature-store.git'
REPO_DIR = 'athletes-mlops-feature-store'

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL

if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

## 2. Install dependencies

Colab needs the project requirements. Locally, if your conda env is already
active with everything installed, you can skip this.


In [ ]:
!pip install -q -r requirements.txt

## 3. Databricks credentials

A freshly cloned repo has no `.env` (it is git-ignored), so enter your
credentials securely here. The token is not printed or saved to the notebook.


In [ ]:
import os
from getpass import getpass

os.environ['MLFLOW_TRACKING_URI'] = 'databricks'
os.environ['DATABRICKS_HOST'] = input('Databricks workspace URL (https://...): ').strip()
os.environ['DATABRICKS_TOKEN'] = getpass('Databricks token: ')
os.environ['MLFLOW_EXPERIMENT_PATH'] = input('MLflow experiment path (/Users/you@school.edu/athletes-mlops-a2): ').strip()
print('Credentials and experiment path set for this session.')

## 4. Run the pipeline

Each cell runs one stage. Run them in order.


In [ ]:
# Stage 1: clean raw data -> processed parquet
!python src/clean.py

In [ ]:
# Stage 2: build the feature source parquet (target, bmi, age_bin)
!python src/featurize.py

In [ ]:
# Stage 3: register the Feast feature definitions
!cd feature_repo && feast apply

In [ ]:
# Stage 4: verify Feast retrieval for both feature versions
!python src/load_features.py

In [ ]:
# Stage 5: run all four experiments (logs to Databricks MLflow)
!python src/run_experiments.py

In [ ]:
# Stage 6: build the comparison summary and charts
!python src/evaluate.py

## 5. View the results

The summary CSV and charts are written to `reports/`.


In [ ]:
import pandas as pd
pd.read_csv('reports/experiment_summary.csv')

In [ ]:
from IPython.display import Image, display
display(Image('reports/metric_comparison.png'))
display(Image('reports/roc_auc_comparison.png'))

All four runs are now visible in your Databricks MLflow experiment, comparable
by their `feature_version` and `C` parameters. Take a screenshot of the
four-run comparison view for the experiment-tracking deliverable.
